In [1]:
import sys
import os


!pip install uv -q


REPO_URL = "https://github.com/PabloVM/JLM-jev_like_model"
REPO_NAME = "JLM-jev_like_model"

!git clone {REPO_URL}


os.chdir("/kaggle/working")


sys.path.append(f"/kaggle/working/{REPO_NAME}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 90.4 MB/s eta 0:00:00
Cloning into 'JLM-jev_like_model'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 30 (delta 8), reused 27 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 230.19 KiB | 6.06 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [2]:

%cd /kaggle/working/JLM-jev_like_model
!git pull

/kaggle/working/JLM-jev_like_model
Already up to date.


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import Qwen3Model

from src.data import JLMTokenizer, JLMDataset, jlm_collate_fn
from src.model import JLM

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:

raw_dataset = load_dataset("tasksource/tasksource-jev-typed-decisions", split="train")


dataset = raw_dataset.filter(lambda example: example["kind"] == "choice")


train_dataset_sample = dataset.select(range(200)) 
print(f"Total examples: {len(train_dataset_sample)}")

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

data/train-00000-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  112MB            

data/train-00000-of-00006.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00006.parquet: reconstructing file:   0%|          |  0.00B /  112MB            

data/train-00001-of-00006.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/999846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/999846 [00:00<?, ? examples/s]

Total examples: 200


In [6]:
train_dataset_sample = train_dataset_sample.filter(lambda example: 1 in example["target"] )

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

In [7]:
tokenizer = JLMTokenizer("Qwen/Qwen3-0.6B")
model = JLM("Qwen/Qwen3-0.6B")
model.to(device)

model.base_model.eval()
for param in model.base_model.parameters():
    param.requires_grad = False

model.head.train()
for param in model.head.parameters():
    param.requires_grad = True

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
print("Base model device:", model.base_model.device)
print("Head device:", next(model.head.parameters()).device)

Base model device: cuda:0
Head device: cuda:0


In [9]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.head.parameters(), lr=1e-3)
train_data = JLMDataset(train_dataset_sample, tokenizer)

train_loader = DataLoader(
    train_data, 
    batch_size=4, 
    shuffle=True, 
    collate_fn=jlm_collate_fn
)




num_epochs = 3

print("--- Training Started ---")
for epoch in range(num_epochs):
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        position_ids = batch["position_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        choice_idx = batch["choices_end_indxs"]
        answer_idx = batch["answer_end_indxs"]

        # Forward pass 
        logits_list = model(
            input_ids=input_ids,
            position_ids=position_ids,
            attention_mask=attention_mask,
            choice_idx=choice_idx,
            answer_idx=answer_idx
        )

        
        batch_loss = torch.tensor(0.0, device=device, requires_grad=True)
        
        for i, logits in enumerate(logits_list):
            item_logits = logits.unsqueeze(0)       # (1, num_choices)
            item_label = labels[i].unsqueeze(0)      # (1,)

            loss_i = criterion(item_logits, item_label)
            batch_loss = batch_loss + loss_i

            
            pred = torch.argmax(item_logits, dim=-1)
            if pred == item_label:
                correct += 1
            total += 1

        batch_loss = batch_loss / len(logits_list)

        # Backpropagation
        batch_loss.backward()
        optimizer.step()

        total_loss += batch_loss.item()

    avg_loss = total_loss / len(train_loader)
    acc = (correct / total) * 100
    print(f"Epoch {epoch + 1}/{num_epochs} | Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")

print("--- Training Finished ---")

--- Training Started ---
Epoch 1/3 | Loss: 407.4353 | Accuracy: 45.73%
Epoch 2/3 | Loss: 113.2357 | Accuracy: 77.39%
Epoch 3/3 | Loss: 118.5008 | Accuracy: 81.41%
--- Training Finished ---
